# PoultryPulse AI — Live Data Integration Test
**Phase: Real Data Pipeline Validation**

This notebook connects to:
- **AGMARKNET API** (data.gov.in) — live mandi broiler prices
- **OpenWeatherMap API** — live Gorakhpur weather (IMD fallback)
- **Supabase** — historical price data + prediction storage

Run cells top-to-bottom. Enter your API keys in Cell 3 (Secrets).

> After the install cell the runtime restarts automatically. Skip Cell 2 on re-run and start from Cell 3.


## 1. GPU & Environment Check

In [ ]:
import subprocess, sys, platform
try:
    gpu = subprocess.check_output(['nvidia-smi'], stderr=subprocess.DEVNULL).decode()
    print('GPU detected:')
    print(gpu)
except Exception:
    print('No GPU — running on CPU (fine for ONNX inference)')
print(f'Python: {sys.version}')
print(f'Platform: {platform.platform()}')


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted at /content/drive')


## 3. Install Dependencies
> This cell restarts the runtime automatically after installing. Skip on re-run.

In [ ]:
import sys, os, subprocess

_INSTALL_FLAG = '/tmp/.poultrypulse_deps_installed'

if os.path.exists(_INSTALL_FLAG):
    print('Dependencies already installed this session. Skipping.')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy'], check=False)

    packages = [
        'numpy==1.26.4', 'pandas==2.2.2', 'scikit-learn==1.6.1',
        'lightgbm==4.3.0', 'statsmodels==0.14.2', 'cmdstanpy==1.2.4',
        'prophet==1.1.5', 'holidays>=0.25', 'onnxruntime==1.19.0',
        'protobuf<5', 'pytest==8.0.0',
        'supabase==2.4.2',      # Supabase Python client
        'requests==2.31.0',     # HTTP for AGMARKNET + OpenWeatherMap
        'python-dotenv==1.0.1', # Load secrets from env
        'tenacity==8.2.3',      # Retry logic for API calls
    ]
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', *packages], check=False)

    if result.returncode != 0:
        print(f'Install failed (exit code {result.returncode}). Check output above.')
        sys.exit(1)

    open(_INSTALL_FLAG, 'w').close()
    print('All packages installed.')
    print('Restarting runtime...')
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)


## 4. Secrets Configuration
Enter your API keys here. These are stored only in memory — never written to disk or Drive.

**Where to get each key:**
- `AGMARKNET_API_KEY` — data.gov.in → My Account → API Keys
- `SUPABASE_URL` — Supabase dashboard → Project Settings → API → Project URL
- `SUPABASE_KEY` — Supabase dashboard → Project Settings → API → anon public key
- `OWM_API_KEY` — openweathermap.org → My API Keys (free tier, backup for IMD)


In [ ]:
# ── Enter your keys here ──────────────────────────────────────────────────
AGMARKNET_API_KEY = 'YOUR_DATA_GOV_IN_API_KEY'   # data.gov.in API key
SUPABASE_URL      = 'https://YOUR_PROJECT.supabase.co'
SUPABASE_KEY      = 'YOUR_SUPABASE_ANON_KEY'      # anon public key
OWM_API_KEY       = 'YOUR_OPENWEATHERMAP_API_KEY'  # free tier

# Gorakhpur coordinates (fixed — do not change)
GORAKHPUR_LAT = 26.7606
GORAKHPUR_LON = 83.3732

# Target mandis per TRD
TARGET_MANDIS = ['Gorakhpur', 'Deoria', 'Basti', 'Kushinagar', 'Maharajganj']

# Validate keys are filled
missing = []
for name, val in [('AGMARKNET_API_KEY', AGMARKNET_API_KEY),
                   ('SUPABASE_URL', SUPABASE_URL),
                   ('SUPABASE_KEY', SUPABASE_KEY),
                   ('OWM_API_KEY', OWM_API_KEY)]:
    if 'YOUR_' in val:
        missing.append(name)
        print(f'  NOT SET: {name}')
    else:
        print(f'  OK: {name}')

if missing:
    print(f'\nFill in {len(missing)} missing key(s) above before continuing.')
else:
    print('\nAll keys configured.')


## 5. Imports & Path Configuration

In [ ]:
import json, pickle, logging, sys, os, time
import numpy as np
import pandas as pd
import requests
from pathlib import Path
from typing import Dict, Any, Tuple, Optional, List
from datetime import datetime, timedelta
from tenacity import retry, stop_after_attempt, wait_exponential
import onnxruntime as ort
from supabase import create_client, Client

# ── Logging ────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True
)
logger = logging.getLogger(__name__)

# ── Paths ──────────────────────────────────────────────────────────────────
DRIVE_PATH  = Path('/content/drive/MyDrive/ColabNotebooks/poutrysense')
MODELS_DIR  = DRIVE_PATH / 'models'
TESTS_DIR   = DRIVE_PATH / 'tests'
TESTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Version check ──────────────────────────────────────────────────────────
print(f'numpy        : {np.__version__}')
print(f'pandas       : {pd.__version__}')
print(f'onnxruntime  : {ort.__version__}')
print(f'Models dir   : {MODELS_DIR} (exists={MODELS_DIR.exists()})')
if MODELS_DIR.exists():
    files = list(MODELS_DIR.iterdir())
    print(f'Model files  : {len(files)} found')
    for f in sorted(files):
        print(f'  {f.name:<30} {f.stat().st_size/1024:>8.1f} KB')


## 6. Model Inference Service

In [ ]:
# ── Prophet module stub (must run before any pickle.load) ─────────────────
from types import ModuleType
_stub = ModuleType('models_inference')
class DummyProphet:
    is_prophet_proxy = True
    def predict(self, df):
        result = df.copy()
        result['yhat']       = 120.0 + np.random.randn(len(df)) * 5
        result['yhat_lower'] = result['yhat'] - 10
        result['yhat_upper'] = result['yhat'] + 10
        return result
_stub.DummyProphet = DummyProphet
for _n in ['models_inference','src.models_inference',
           'scripts.models_inference','colab_package.models_inference']:
    sys.modules[_n] = _stub
setattr(sys.modules['__main__'], 'DummyProphet', DummyProphet)

class ModelInferenceService:
    """Loads and runs the full PoultryPulse ensemble."""

    # LightGBM ONNX was exported with 200 features (proxy model).
    # When Kiro exports the real trained LightGBM on 45 features,
    # update N_LGB_FEATURES to 45.
    N_LGB_FEATURES = 200

    def __init__(self, models_dir=None):
        self.models_dir        = Path(models_dir) if models_dir else MODELS_DIR
        self.models            = {}
        self.onnx_sessions     = {}
        self.conformal_scalars = {}
        self._load_report      = []

    def load_models(self) -> bool:
        logger.info(f'Loading models from: {self.models_dir}')
        if not self.models_dir.exists():
            logger.error('Models directory not found')
            return False

        for key, fname in [('arima','arima_model.pkl'),
                            ('prophet','prophet_model.pkl'),
                            ('ensemble_ridge','ridge_meta.pkl')]:
            p = self.models_dir / fname
            if not p.exists():
                self._load_report.append((key,'missing')); continue
            try:
                with open(p,'rb') as f:
                    obj = pickle.load(f)
                if key == 'ensemble_ridge' and isinstance(obj, dict):
                    raise TypeError('ridge_meta.pkl contains dict, not Ridge')
                self.models[key] = obj
                self._load_report.append((key,'ok'))
                logger.info(f'  {key} loaded — {type(obj).__name__}')
            except Exception as e:
                self._load_report.append((key, f'error: {e}'))
                logger.warning(f'  {key} failed: {e}')

        for key, fname in [('lightgbm','lightgbm.onnx'),
                            ('tft','tft_quantized.onnx')]:
            p = self.models_dir / fname
            if not p.exists():
                self._load_report.append((key,'missing')); continue
            try:
                sess = ort.InferenceSession(str(p), providers=['CPUExecutionProvider'])
                self.onnx_sessions[key] = sess
                inp = sess.get_inputs()[0]
                self._load_report.append((key,'ok'))
                logger.info(f'  {key} ONNX loaded — input shape: {inp.shape}')
            except Exception as e:
                self._load_report.append((key, f'error: {e}'))
                logger.warning(f'  {key} ONNX failed: {e}')

        p = self.models_dir / 'calibration_scalars.json'
        try:
            with open(p) as f:
                self.conformal_scalars['ensemble'] = json.load(f).get('q_hat', 0.0)
            self._load_report.append(('conformal','ok'))
            logger.info(f'  conformal q_hat: {self.conformal_scalars["ensemble"]}')
        except Exception as e:
            self.conformal_scalars['ensemble'] = 0.0
            self._load_report.append(('conformal', f'error: {e}'))

        ok = sum(1 for _, s in self._load_report if s == 'ok')
        logger.info(f'Load complete: {ok}/{len(self._load_report)}')
        return ok > 0

    def predict_base_models(self, X_features, tft_features=None):
        n     = len(X_features)
        preds = np.zeros((n, 4), dtype=np.float32)

        if 'arima' in self.models:
            try:
                raw = self.models['arima'].forecast(steps=n)
                preds[:,0] = np.asarray(raw, dtype=np.float32).flatten()
            except Exception as e:
                logger.warning(f'ARIMA failed: {e}')

        if 'prophet' in self.models:
            try:
                future = pd.DataFrame({'ds': pd.date_range(
                    start=pd.Timestamp.today().normalize(), periods=n, freq='D')})
                out = self.models['prophet'].predict(future)
                preds[:,1] = out['yhat'].values.astype(np.float32)
            except Exception as e:
                logger.warning(f'Prophet failed: {e}')

        if 'lightgbm' in self.onnx_sessions:
            try:
                sess     = self.onnx_sessions['lightgbm']
                inp_name = sess.get_inputs()[0].name
                expected = sess.get_inputs()[0].shape[1]
                X_in = X_features[:, :expected].astype(np.float32)
                if X_in.shape[1] < expected:
                    pad  = np.zeros((n, expected - X_in.shape[1]), dtype=np.float32)
                    X_in = np.hstack([X_in, pad])
                out = sess.run(None, {inp_name: X_in})
                preds[:,2] = out[0].flatten().astype(np.float32)
            except Exception as e:
                logger.warning(f'LightGBM failed: {e}')

        if 'tft' in self.onnx_sessions and tft_features is not None:
            try:
                sess = self.onnx_sessions['tft']
                ort_inputs = {
                    inp.name: tft_features[inp.name].astype(np.float32)
                    for inp in sess.get_inputs() if inp.name in tft_features
                }
                out = sess.run(None, ort_inputs)
                preds[:,3] = out[0].flatten().astype(np.float32)
            except Exception as e:
                logger.warning(f'TFT failed: {e}')

        return preds

    def apply_conformal_bounds(self, p50):
        q = float(self.conformal_scalars.get('ensemble', 0.0))
        return np.maximum(p50 - q, 0.0), p50 + q

    def predict(self, X_features, tft_features=None):
        base = self.predict_base_models(X_features, tft_features)
        if 'ensemble_ridge' not in self.models:
            logger.warning('Ridge not loaded — using column mean')
            p50 = base.mean(axis=1)
        else:
            p50 = self.models['ensemble_ridge'].predict(base)
        p10, p90 = self.apply_conformal_bounds(p50)
        return {
            'p10': p10.tolist(), 'p50': p50.tolist(), 'p90': p90.tolist(),
            'base_model_predictions': base.tolist(),
            'q_hat_applied': self.conformal_scalars.get('ensemble', 0.0),
        }

print('ModelInferenceService defined.')


## 7. Load Models

In [ ]:
service = ModelInferenceService()
success = service.load_models()

print('\n-- Load Report --')
for component, status in service._load_report:
    icon = 'OK' if status == 'ok' else ('MISSING' if status == 'missing' else 'FAIL')
    print(f'  [{icon}] {component:20s} : {status}')

loaded = sum(1 for _, s in service._load_report if s == 'ok')
total  = len(service._load_report)
print(f'\n{loaded}/{total} components loaded.')


## 8. Live Data — AGMARKNET API
Fetches today's broiler price from the AGMARKNET API (data.gov.in).
Falls back to the last known price from Supabase if the API is unavailable.


In [ ]:
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def fetch_agmarknet_price(mandi: str, api_key: str) -> Optional[Dict]:
    """
    Fetch today's broiler price from AGMARKNET API.
    Endpoint: data.gov.in/resource (AGMARKNET dataset)
    Returns dict with date, mandi_name, broiler_price_per_kg, arrivals_kg
    """
    # AGMARKNET commodity code for broiler/chicken
    COMMODITY = 'Broiler(Chicken)'
    STATE      = 'Uttar Pradesh'

    today = datetime.now().strftime('%d/%m/%Y')

    url = 'https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070'
    params = {
        'api-key'  : api_key,
        'format'   : 'json',
        'limit'    : '10',
        'filters[state]'    : STATE,
        'filters[market]'   : mandi,
        'filters[commodity]': COMMODITY,
        'filters[arrival_date]': today,
    }

    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    records = data.get('records', [])
    if not records:
        return None

    rec = records[0]
    modal_price = float(rec.get('modal_price', 0))

    # Validate price range per TRD: Rs 80-250/kg
    if not (80 <= modal_price <= 250):
        logger.warning(f'Price {modal_price} outside valid range 80-250 for {mandi}')
        return None

    return {
        'date'                : rec.get('arrival_date', today),
        'mandi_name'          : mandi,
        'broiler_price_per_kg': modal_price,
        'arrivals_kg'         : float(rec.get('arrivals', 0)),
        'source'              : 'agmarknet_live',
    }


def fetch_all_mandis(api_key: str) -> pd.DataFrame:
    """Fetch today's price for all 5 target mandis."""
    results = []
    print(f'Fetching live prices for {len(TARGET_MANDIS)} mandis...')

    for mandi in TARGET_MANDIS:
        try:
            rec = fetch_agmarknet_price(mandi, api_key)
            if rec:
                results.append(rec)
                print(f'  {mandi:<15} Rs {rec["broiler_price_per_kg"]:.2f}/kg  arrivals={rec["arrivals_kg"]:.0f}kg')
            else:
                print(f'  {mandi:<15} No data today (market may be closed or API returning empty)')
        except Exception as e:
            print(f'  {mandi:<15} API error: {e}')

    if results:
        return pd.DataFrame(results)
    else:
        print('\nNo live data returned. Will use Supabase historical fallback.')
        return pd.DataFrame()


# ── Run the fetch ──────────────────────────────────────────────────────────
if 'YOUR_' in AGMARKNET_API_KEY:
    print('AGMARKNET_API_KEY not set — skipping live fetch.')
    print('To test: add your data.gov.in API key in Cell 4 (Secrets).')
    live_df = pd.DataFrame()
else:
    live_df = fetch_all_mandis(AGMARKNET_API_KEY)
    print(f'\nLive records fetched: {len(live_df)}')
    if not live_df.empty:
        print(live_df.to_string(index=False))


## 9. Live Data — Weather (OpenWeatherMap)
Fetches current weather for Gorakhpur. Used for `temperature_max`, `heat_stress_7d`, and `monsoon_phase` features.
Primary source is IMD API; OpenWeatherMap is the configured fallback per the training guide.


In [ ]:
def fetch_weather(api_key: str, lat: float, lon: float) -> Dict:
    """
    Fetch current weather from OpenWeatherMap free tier.
    Gorakhpur lat/lon: 26.7606, 83.3732
    Returns feature-ready dict matching the 45-feature schema.
    """
    url = 'https://api.openweathermap.org/data/2.5/weather'
    params = {
        'lat'   : lat,
        'lon'   : lon,
        'appid' : api_key,
        'units' : 'metric',  # Celsius
    }
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    d = resp.json()

    temp_max  = d['main']['temp_max']
    temp_min  = d['main']['temp_min']
    humidity  = d['main']['humidity']
    rain_1h   = d.get('rain', {}).get('1h', 0.0)
    wind_ms   = d['wind']['speed']

    # Monsoon phase from current month (simplified)
    month = datetime.now().month
    if month in [3, 4, 5]:   monsoon_phase = 0  # pre-monsoon
    elif month in [6]:        monsoon_phase = 1  # active onset
    elif month in [7, 8]:     monsoon_phase = 2  # peak
    elif month in [9, 10]:    monsoon_phase = 3  # retreat
    else:                     monsoon_phase = 0  # dry

    # Heat stress: temp > 35C counts as heat stress day
    heat_stress_today = 1 if temp_max > 35 else 0

    return {
        'temperature_max'    : temp_max,
        'temperature_min'    : temp_min,
        'humidity_pct'       : humidity,
        'rainfall_1h_mm'     : rain_1h,
        'wind_speed_ms'      : wind_ms,
        'heat_stress_today'  : heat_stress_today,
        'monsoon_phase'      : monsoon_phase,
        'weather_source'     : 'openweathermap',
        'fetched_at'         : datetime.now().isoformat(),
    }


if 'YOUR_' in OWM_API_KEY:
    print('OWM_API_KEY not set — using default weather values for Gorakhpur.')
    weather_data = {
        'temperature_max' : 35.0,
        'temperature_min' : 26.0,
        'humidity_pct'    : 65.0,
        'rainfall_1h_mm'  : 0.0,
        'wind_speed_ms'   : 2.5,
        'heat_stress_today': 0,
        'monsoon_phase'   : 1,
        'weather_source'  : 'default_gorakhpur_may',
        'fetched_at'      : datetime.now().isoformat(),
    }
else:
    try:
        weather_data = fetch_weather(OWM_API_KEY, GORAKHPUR_LAT, GORAKHPUR_LON)
        print('Live weather fetched:')
    except Exception as e:
        print(f'Weather API error: {e} — using defaults')
        weather_data = {
            'temperature_max': 35.0, 'temperature_min': 26.0,
            'humidity_pct': 65.0, 'rainfall_1h_mm': 0.0,
            'wind_speed_ms': 2.5, 'heat_stress_today': 0,
            'monsoon_phase': 1, 'weather_source': 'default_fallback',
            'fetched_at': datetime.now().isoformat(),
        }

for k, v in weather_data.items():
    print(f'  {k:<25}: {v}')


## 10. Supabase — Historical Price Data
Fetches the last 90 days of Gorakhpur broiler prices from Supabase `raw_prices` table.
This data is used to compute lag features (price_lag_1d through price_lag_42d) and rolling statistics.

**Expected Supabase table schema:**
```sql
-- raw_prices table
CREATE TABLE raw_prices (
    id                   BIGSERIAL PRIMARY KEY,
    date                 DATE NOT NULL,
    mandi_name           TEXT NOT NULL,
    broiler_price_per_kg NUMERIC(8,2),
    arrivals_kg          NUMERIC(12,2),
    source               TEXT,
    created_at           TIMESTAMPTZ DEFAULT NOW()
);
```


In [ ]:
def connect_supabase(url: str, key: str) -> Optional[Client]:
    """Connect to Supabase and verify connection.""",
    if 'YOUR_' in url or 'YOUR_' in key:
        return None
    try:
        client = create_client(url, key)
        return client
    except Exception as e:
        logger.error(f'Supabase connection failed: {e}')
        return None


def fetch_historical_prices(client: Client, mandi: str = 'Gorakhpur',
                             days: int = 90) -> pd.DataFrame:
    """
    Fetch last N days of prices from Supabase raw_prices table.
    Returns DataFrame sorted by date ascending (oldest first).
    """
    cutoff = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    resp = (
        client.table('raw_prices')
        .select('date, mandi_name, broiler_price_per_kg, arrivals_kg, source')
        .eq('mandi_name', mandi)
        .gte('date', cutoff)
        .order('date', desc=False)
        .execute()
    )
    if not resp.data:
        return pd.DataFrame()

    df = pd.DataFrame(resp.data)
    df['date'] = pd.to_datetime(df['date'])
    df['broiler_price_per_kg'] = pd.to_numeric(df['broiler_price_per_kg'], errors='coerce')
    # Validate price range per TRD
    df = df[df['broiler_price_per_kg'].between(80, 250)]
    return df.sort_values('date').reset_index(drop=True)


def fetch_latest_prediction(client: Client, mandi: str = 'Gorakhpur') -> Optional[Dict]:
    """Fetch the most recent stored prediction for this mandi.""",
    try:
        resp = (
            client.table('predictions')
            .select('*')
            .eq('mandi_name', mandi)
            .order('forecast_date', desc=True)
            .limit(1)
            .execute()
        )
        return resp.data[0] if resp.data else None
    except Exception as e:
        logger.warning(f'Could not fetch latest prediction: {e}')
        return None


# ── Connect and fetch ──────────────────────────────────────────────────────
supabase_client = connect_supabase(SUPABASE_URL, SUPABASE_KEY)

if supabase_client is None:
    print('Supabase not configured — using synthetic historical data.')
    # Generate 90 days of synthetic historical data as fallback
    dates = pd.date_range(end=pd.Timestamp.today().normalize(), periods=90, freq='D')
    np.random.seed(42)
    historical_df = pd.DataFrame({
        'date'                : dates,
        'mandi_name'          : 'Gorakhpur',
        'broiler_price_per_kg': 145 + np.cumsum(np.random.randn(90) * 1.5),
        'arrivals_kg'         : 40000 + np.random.randn(90) * 5000,
        'source'              : 'synthetic',
    })
    historical_df['broiler_price_per_kg'] = historical_df['broiler_price_per_kg'].clip(80, 250)
    print(f'Synthetic historical data: {len(historical_df)} rows')
else:
    print('Supabase connected.')
    historical_df = fetch_historical_prices(supabase_client, mandi='Gorakhpur', days=90)

    if historical_df.empty:
        print('No historical data in raw_prices table yet.')
        print('The table exists but has no rows — upload your training data first.')
        # Fall back to synthetic
        dates = pd.date_range(end=pd.Timestamp.today().normalize(), periods=90, freq='D')
        np.random.seed(42)
        historical_df = pd.DataFrame({
            'date': dates, 'mandi_name': 'Gorakhpur',
            'broiler_price_per_kg': (145 + np.cumsum(np.random.randn(90)*1.5)).clip(80, 250),
            'arrivals_kg': 40000 + np.random.randn(90)*5000, 'source': 'synthetic',
        })
    else:
        print(f'Historical data loaded: {len(historical_df)} rows')
        print(f'Date range: {historical_df["date"].min().date()} to {historical_df["date"].max().date()}')
        print(f'Price range: Rs {historical_df["broiler_price_per_kg"].min():.2f} - Rs {historical_df["broiler_price_per_kg"].max():.2f}')
        print(historical_df.tail(5).to_string(index=False))

    prev_pred = fetch_latest_prediction(supabase_client, 'Gorakhpur')
    if prev_pred:
        print(f'\nLast stored prediction: P50=Rs {prev_pred.get("p50"):.2f}  date={prev_pred.get("forecast_date")}')


## 11. Feature Engineering (45 Features from Live + Historical Data)
Computes all 45 features from TRD §4.3 using the live price + weather data fetched above.


In [ ]:
# UP festival calendar (from Training Guide Section 4.3)
UP_FESTIVALS_2026 = [
    datetime(2026, 3, 25),   # Holi
    datetime(2026, 3, 31),   # Eid-ul-Fitr (approx)
    datetime(2026, 4, 2),    # Ram Navami
    datetime(2026, 4, 14),   # Baisakhi
    datetime(2026, 6, 7),    # Eid-ul-Adha (approx)
    datetime(2026, 8, 9),    # Muharram (approx)
    datetime(2026, 8, 19),   # Raksha Bandhan
    datetime(2026, 10, 2),   # Navratri start
    datetime(2026, 10, 20),  # Diwali
    datetime(2026, 12, 25),  # Christmas
    datetime(2026, 12, 31),  # New Year
]
# Navratri causes price DROP — separate flag
NAVRATRI_2026 = [datetime(2026, 4, 6), datetime(2026, 10, 2)]

def festival_7d_flag(target_date: datetime) -> int:
    for f in UP_FESTIVALS_2026:
        if abs((target_date - f).days) <= 7:
            return 1
    return 0

def days_to_next_festival(target_date: datetime) -> int:
    future = [f for f in UP_FESTIVALS_2026 if f >= target_date]
    if not future:
        return 365
    return (min(future) - target_date).days

def build_feature_vector(
    historical_prices: pd.Series,   # indexed by date, values = price
    target_date: datetime,
    weather: Dict,
    n_features: int = 200            # pad to match LightGBM ONNX input shape
) -> np.ndarray:
    """
    Build the 45-feature vector for a single prediction date.
    Pads to n_features with zeros so LightGBM ONNX accepts it.
    """
    prices = historical_prices.copy()
    prices.index = pd.to_datetime(prices.index)
    prices = prices.sort_index()

    def lag(n):
        d = target_date - timedelta(days=n)
        # Find nearest available price within 3 days
        for offset in range(4):
            key = d - timedelta(days=offset)
            if key in prices.index:
                return float(prices[key])
        return float(prices.iloc[-1]) if len(prices) > 0 else 150.0

    def rolling_mean(n):
        cutoff = target_date - timedelta(days=n)
        window = prices[prices.index >= cutoff]
        return float(window.mean()) if len(window) > 0 else 150.0

    def rolling_std(n):
        cutoff = target_date - timedelta(days=n)
        window = prices[prices.index >= cutoff]
        return float(window.std()) if len(window) > 1 else 5.0

    # ── 45 features matching TRD §4.3 ─────────────────────────────────────
    p1  = lag(1);  p3  = lag(3);  p7  = lag(7)
    p14 = lag(14); p21 = lag(21); p30 = lag(30); p42 = lag(42)

    ma7   = rolling_mean(7);  ma14  = rolling_mean(14);  ma30 = rolling_mean(30)
    std7  = rolling_std(7);   std30 = rolling_std(30)
    mom14 = p1 - p14  # price momentum 14d
    slope = (p1 - p14) / 14.0  # trend slope

    # Feed cost (use proxies — replace with real NCDEX data when available)
    maize_current   = 2200.0   # Rs/quintal — update from NCDEX
    soy_current     = 4800.0   # Rs/quintal — update from NCDEX
    maize_lag42     = 2150.0   # maize price 42 days ago
    soy_lag42       = 4700.0
    palm_oil_lag42  = 950.0
    feed_cost_ratio = p1 / (maize_lag42 / 100) if maize_lag42 > 0 else 1.0

    month = target_date.month
    dow   = target_date.weekday()  # 0=Monday
    features_45 = [
        # Price lags (7)
        p1, p3, p7, p14, p21, p30, p42,
        # Rolling stats (7)
        ma7, ma14, ma30, std7, std30, mom14, slope,
        # Feed cost (5)
        feed_cost_ratio, soy_lag42, palm_oil_lag42, maize_current, soy_current,
        # Weather (6)
        weather.get('temperature_max', 35.0),
        weather.get('temperature_min', 26.0),
        weather.get('heat_stress_today', 0),
        1 if weather.get('temperature_min', 26) < 10 else 0,  # cold_wave
        weather.get('rainfall_1h_mm', 0.0) * 24,              # rainfall_7d_mm proxy
        weather.get('monsoon_phase', 0),
        # Disease (2)
        0, 0,   # hpai_district_flag, hpai_adjacent — default safe
        # Festival & calendar (7)
        festival_7d_flag(target_date),
        days_to_next_festival(target_date),
        1 if dow >= 5 else 0,                    # weekend_flag
        np.sin(2 * np.pi * month / 12),          # month_sin
        np.cos(2 * np.pi * month / 12),          # month_cos
        np.sin(2 * np.pi * dow / 7),             # day_of_week_sin
        np.cos(2 * np.pi * dow / 7),             # day_of_week_cos
        # Market demand signals (4)
        0.0, 0.0, 0.0, 1.0,  # necc_delta, egg_chg, google_trends, egg_prod_index
        # Supply signals (3)
        0.0, 0.0, 0,          # doc_placement_lag42, fuel_delta, transport_flag
        # External market (3)
        0.0, 0.0, 1.0,        # ncdex_spread, mcx_palm_delta, mla_global
        # Derived interaction (2)
        feed_cost_ratio * weather.get('heat_stress_today', 0),  # feed_weather_stress
        0,                                                        # festival_hpai_overlap
    ]
    assert len(features_45) == 45, f'Got {len(features_45)} features, expected 45'

    # Pad to n_features for LightGBM ONNX compatibility
    padded = features_45 + [0.0] * (n_features - 45)
    return np.array(padded[:n_features], dtype=np.float32)


# ── Build feature matrix for 7-day forecast horizon ───────────────────────
price_series = historical_df.set_index('date')['broiler_price_per_kg']

# If we have live data for today, append it
if not live_df.empty:
    gorakhpur_live = live_df[live_df['mandi_name'] == 'Gorakhpur']
    if not gorakhpur_live.empty:
        today_price = gorakhpur_live.iloc[0]['broiler_price_per_kg']
        today_date  = pd.Timestamp.today().normalize()
        price_series[today_date] = today_price
        print(f'Live price appended to series: Rs {today_price:.2f} on {today_date.date()}')

price_series = price_series.sort_index()

FORECAST_DAYS = 7
X_live = np.zeros((FORECAST_DAYS, 200), dtype=np.float32)

for i in range(FORECAST_DAYS):
    target_dt = datetime.now() + timedelta(days=i+1)
    X_live[i] = build_feature_vector(price_series, target_dt, weather_data, n_features=200)

print(f'\nFeature matrix built: {X_live.shape}  (days x features)')
print(f'Feature range: [{X_live.min():.3f}, {X_live.max():.3f}]')
print(f'Non-zero features per row: {(X_live != 0).sum(axis=1).tolist()}')


## 12. Run Ensemble Prediction on Live Features

In [ ]:
print('=== 7-Day Poultry Price Forecast (Gorakhpur) ===\n')

predictions = service.predict(X_live, tft_features=None)

forecast_dates = pd.date_range(
    start=pd.Timestamp.today().normalize() + pd.Timedelta(days=1),
    periods=FORECAST_DAYS, freq='D'
)

results_df = pd.DataFrame({
    'Day'           : range(1, FORECAST_DAYS + 1),
    'Date'          : forecast_dates.date,
    'P10 (Rs/kg)'  : [round(v, 2) for v in predictions['p10']],
    'P50 (Rs/kg)'  : [round(v, 2) for v in predictions['p50']],
    'P90 (Rs/kg)'  : [round(v, 2) for v in predictions['p90']],
    'Range (Rs)'   : [round(hi-lo, 2) for lo,hi in zip(predictions['p10'], predictions['p90'])],
})

print('Broiler Price Forecast — Gorakhpur Mandi (Rs/kg):')
print(results_df.to_string(index=False))
print(f'\nConformal q_hat applied : Rs {predictions["q_hat_applied"]}')
print(f'p10 <= p50 <= p90 valid : '
      f'{all(lo<=mid<=hi for lo,mid,hi in zip(predictions["p10"],predictions["p50"],predictions["p90"]))}')

print('\nBase model contributions:')
base = np.array(predictions['base_model_predictions'])
for i, label in enumerate(['ARIMA','Prophet','LightGBM','TFT']):
    col    = base[:, i]
    active = not np.allclose(col, 0)
    status = f'mean=Rs {col.mean():.2f}  std={col.std():.2f}' if active else 'zero (model contributing 0)'
    print(f'  {label:<12}: {status}')


## 13. Save Predictions to Supabase
Writes the 7-day forecast to the `predictions` table in Supabase.

**Expected table schema:**
```sql
CREATE TABLE predictions (
    id             BIGSERIAL PRIMARY KEY,
    forecast_date  DATE NOT NULL,
    mandi_name     TEXT NOT NULL,
    p10            NUMERIC(8,2),
    p50            NUMERIC(8,2),
    p90            NUMERIC(8,2),
    q_hat          NUMERIC(8,2),
    model_version  TEXT,
    created_at     TIMESTAMPTZ DEFAULT NOW()
);
```


In [ ]:
def save_predictions_to_supabase(client: Client, results: pd.DataFrame,
                                   mandi: str, q_hat: float) -> bool:
    """Upsert 7-day forecast rows into Supabase predictions table.""",
    rows = []
    for _, row in results.iterrows():
        rows.append({
            'forecast_date' : str(row['Date']),
            'mandi_name'    : mandi,
            'p10'           : row['P10 (Rs/kg)'],
            'p50'           : row['P50 (Rs/kg)'],
            'p90'           : row['P90 (Rs/kg)'],
            'q_hat'         : q_hat,
            'model_version' : 'ensemble_v1_dummy',
        })

    resp = (
        client.table('predictions')
        .upsert(rows, on_conflict='forecast_date,mandi_name')
        .execute()
    )
    return bool(resp.data)


def save_live_price_to_supabase(client: Client, live_prices: pd.DataFrame) -> bool:
    """Upsert today's live prices into Supabase raw_prices table.""",
    if live_prices.empty:
        return False
    rows = live_prices[['date','mandi_name','broiler_price_per_kg',
                          'arrivals_kg','source']].to_dict('records')
    resp = (
        client.table('raw_prices')
        .upsert(rows, on_conflict='date,mandi_name')
        .execute()
    )
    return bool(resp.data)


if supabase_client is None:
    print('Supabase not configured — predictions not saved.')
    print('Add SUPABASE_URL and SUPABASE_KEY in Cell 4 to enable saving.')
else:
    # Save live prices
    if not live_df.empty:
        ok = save_live_price_to_supabase(supabase_client, live_df)
        print(f'Live prices saved to raw_prices: {ok}')
    else:
        print('No live prices to save (API not configured or market closed)')

    # Save predictions
    ok = save_predictions_to_supabase(
        supabase_client, results_df, 'Gorakhpur', predictions['q_hat_applied'])
    print(f'Predictions saved to predictions table: {ok}')

    if ok:
        print('\nVerifying saved predictions:')
        saved = fetch_latest_prediction(supabase_client, 'Gorakhpur')
        if saved:
            print(f'  Latest forecast: P50=Rs {saved.get("p50")} on {saved.get("forecast_date")}')


## 14. Export Test Results

In [ ]:
from datetime import datetime as _dt
output_path = TESTS_DIR / f'live_test_{_dt.now().strftime("%Y%m%d_%H%M%S")}.json'

summary = {
    'timestamp'         : _dt.now().isoformat(),
    'data_sources'      : {
        'agmarknet_live'  : not live_df.empty,
        'weather_live'    : weather_data.get('weather_source') not in ('default_gorakhpur_may','default_fallback'),
        'supabase'        : supabase_client is not None,
        'historical_rows' : len(historical_df),
    },
    'model_load_report' : [{'component':k,'status':s} for k,s in service._load_report],
    'forecast_gorakhpur': results_df.to_dict('records'),
    'q_hat_applied'     : predictions['q_hat_applied'],
    'p10_lte_p50_lte_p90': all(
        lo<=mid<=hi for lo,mid,hi in zip(
            predictions['p10'], predictions['p50'], predictions['p90'])),
}

with open(output_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f'Results saved to: {output_path}')
print(json.dumps(summary, indent=2, default=str))


## 15. Summary

In [ ]:
print('=' * 62)
print('POULTRYPULSE AI — LIVE DATA INTEGRATION TEST SUMMARY')
print('=' * 62)

# Data sources
agmarknet_ok = not live_df.empty
weather_ok   = 'default' not in weather_data.get('weather_source','')
supabase_ok  = supabase_client is not None

for label, ok, hint in [
    ('AGMARKNET live price', agmarknet_ok,
     'Add AGMARKNET_API_KEY in Cell 4' if not agmarknet_ok else f'{len(live_df)} mandi records'),
    ('Weather (OWM)',        weather_ok,
     'Add OWM_API_KEY in Cell 4'        if not weather_ok   else 'Gorakhpur live weather'),
    ('Supabase connected',   supabase_ok,
     'Add SUPABASE_URL + KEY in Cell 4' if not supabase_ok  else 'Historical + predictions'),
]:
    icon = 'OK' if ok else 'NOT CONFIGURED'
    print(f'  [{icon:14s}] {label:<30} {hint}')

# Models
loaded = sum(1 for _, s in service._load_report if s == 'ok')
print(f'\n  [OK            ] Models loaded: {loaded}/6')

# Forecast
valid_intervals = all(
    lo<=mid<=hi for lo,mid,hi in zip(
        predictions['p10'], predictions['p50'], predictions['p90']))
print(f'  [{"OK" if valid_intervals else "FAIL":14s}] p10<=p50<=p90 valid intervals')

p50_mean = sum(predictions["p50"]) / len(predictions["p50"])
print(f'  [INFO          ] Mean P50 forecast: Rs {p50_mean:.2f}/kg')

print('\n  Next steps:')
print('  1. Add your 3 API keys in Cell 4 to enable live data')
print('  2. Upload training data to Supabase raw_prices table')
print('  3. Once real models are trained by Kiro, swap dummy')
print('     model files with real .pkl / .onnx files in Drive')
print('  4. Re-run this notebook daily to validate live pipeline')
print('=' * 62)
